In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!pip install --default-timeout=200 google-adk

In [4]:
import os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ["GOOGLE_API_KEY"] = secrets.get_secret("GOOGLE_API_KEY")

import asyncio
from typing import Optional
from google.adk.agents import LlmAgent
from google.adk.agents.callback_context import CallbackContext
from google.adk.models import LlmResponse, LlmRequest
from google.adk.runners import InMemoryRunner
from google.genai import types

MODEL = "gemini-2.5-flash-lite"


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


In [5]:
# %% CELL 3 — Tools: the "silent correction" mechanism
# The interview agent calls these quietly, mid-roleplay, without breaking character.
 
def log_error(error_type: str, original_text: str, suggestion: str) -> dict:
    """Silently records a language error the learner made, for later feedback.
 
    Args:
        error_type: category, e.g. 'article', 'verb_tense', 'word_choice', 'register'
        original_text: what the learner actually said
        suggestion: the corrected/more natural version
    Returns:
        confirmation dict
    """
    # NOTE: ADK auto-injects tool_context if we add it as a param; kept simple here.
    # We append to a module-level list and merge into session state in the callback below.
    _ERROR_BUFFER.append(
        {"type": error_type, "original": original_text, "suggestion": suggestion}
    )
    return {"status": "logged"}
 
_ERROR_BUFFER = []  # simple in-memory buffer for this demo; flushed into session.state
 
 
def get_error_summary() -> dict:
    """Returns all language errors logged so far in this practice session."""
    return {"errors": _ERROR_BUFFER, "count": len(_ERROR_BUFFER)}
 

In [6]:
%%writefile interview_mcp_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("interview-tools")

_ERROR_BUFFER = []

@mcp.tool()
def log_error(error_type: str, original_text: str, suggestion: str) -> dict:
    """Silently records a language error the learner made, for later feedback.

    Args:
        error_type: category, e.g. 'article', 'verb_tense', 'word_choice', 'register'
        original_text: what the learner actually said
        suggestion: the corrected/more natural version
    """
    _ERROR_BUFFER.append(
        {"type": error_type, "original": original_text, "suggestion": suggestion}
    )
    return {"status": "logged", "count": len(_ERROR_BUFFER)}

@mcp.tool()
def get_error_summary() -> dict:
    """Returns all language errors logged so far in this practice session."""
    return {"errors": _ERROR_BUFFER, "count": len(_ERROR_BUFFER)}

if __name__ == "__main__":
    mcp.run(transport="stdio")

Overwriting interview_mcp_server.py


In [7]:
# %% CELL 4 — Guardrail: break character if the learner asks something real
REAL_WORLD_TRIGGERS = [
    "is this a real job", "should i actually take", "is this offer real",
    "give me real legal advice", "actual salary negotiation for my real job",
    "this isn't practice", "i have a real interview tomorrow and need advice on my actual",
]
 
def breakout_guardrail(
    callback_context: CallbackContext, llm_request: LlmRequest
) -> Optional[LlmResponse]:
    """before_model_callback: detects when the learner is asking a genuine
    real-world question instead of practicing, and breaks character safely."""
    last_user_text = ""
    if llm_request.contents:
        last_part = llm_request.contents[-1]
        if last_part.parts:
            last_user_text = (last_part.parts[0].text or "").lower()
 
    if any(trigger in last_user_text for trigger in REAL_WORLD_TRIGGERS):
        return LlmResponse(
            content=types.Content(
                role="model",
                parts=[types.Part(text=(
                    "Let's pause the roleplay for a second — that sounded like a real "
                    "question, not practice. I can't give real hiring, legal, or salary "
                    "advice here. If you'd like, we can resume the practice interview, "
                    "or you can ask a career counselor or trusted advisor about the real "
                    "situation. Want to continue practicing?"
                ))],
            )
        )
    return None  # allow the model call to proceed normally
 
 

In [8]:
import sys
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset as MCPToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters

interview_mcp_toolset = MCPToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command=sys.executable,
            args=["interview_mcp_server.py"],
        ),
        timeout=30,
    ),
)

In [9]:

# %% CELL 5 — Agent 1: the interview practice agent (roleplay + silent correction)
interview_agent = LlmAgent(
    name="InterviewCoach",
    model=MODEL,
    description="Runs realistic mock job interviews and quietly logs language errors.",
    instruction="""
You are a friendly but professional hiring manager conducting a MOCK job interview
with an ESL (English as a Second Language) learner, to help them practice.
 
Rules:
1. Stay in character as the interviewer. Ask one realistic interview question at a
   time (e.g. "Tell me about yourself", "Why do you want this role",
   "Describe a challenge you overcame"). Wait for their answer before continuing.
2. Whenever the learner's answer contains a grammar, word-choice, or register error,
   call the log_error tool with the error type, their original phrasing, and a
   corrected suggestion. Do this SILENTLY — never mention the correction in your
   reply, never break character to point it out mid-interview.
   2b. Never write out your reasoning, tool code, or planning process as part of your
    reply. Your reply must ONLY be natural spoken interview dialogue — as if you
    are actually speaking out loud to a real candidate. No code blocks, no
    explanations of what you're doing, no "thought" text.
2c. Every single turn, you must ALWAYS include a normal spoken reply to the
    candidate — even if you're also logging an error in the background. Never
    send back an empty response.
3. Keep your own replies natural and encouraging, like a real interviewer would be.
4. Ask 4-5 questions total, then say: "That's great, thanks for your time today —
   want me to wrap up with some feedback?"
5. Never give real hiring decisions, real salary numbers, or real legal advice —
   this is always practice.
""",
    tools=[interview_mcp_toolset],
    before_model_callback=breakout_guardrail,
)

In [10]:

# %% CELL 6 — Agent 2: the progress / feedback agent
progress_agent = LlmAgent(
    name="ProgressCoach",
    model=MODEL,
    description="Delivers an encouraging end-of-session feedback report on language errors.",
    instruction="""
You give warm, specific, encouraging feedback to an ESL learner after a mock
interview practice session.
 
Call get_error_summary to retrieve the logged errors, then:
1. Group errors by type (article, verb tense, word choice, register, etc.)
2. Pick the 2-3 most important patterns — not every single error, focus on what
   matters most for sounding natural in a real interview.
3. For each pattern, show one real example from their answers and the corrected
   version.
4. End with one specific thing they did well.
Keep the tone like a supportive tutor, never harsh or clinical.
""",
    tools=[interview_mcp_toolset],
)
 

In [11]:

# %% CELL 7 — Root orchestrator: delegates between practice and feedback
root_agent = LlmAgent(
    name="ESLPracticeOrchestrator",
    model=MODEL,
    description="Routes between mock interview practice and end-of-session feedback.",
    instruction="""
You manage an ESL interview-practice session. If the learner wants to practice or
continue an interview, delegate to InterviewCoach. If they ask for feedback, a
summary, or say they're done, delegate to ProgressCoach. Start every new
conversation by delegating to InterviewCoach to begin the mock interview.
""",
    sub_agents=[interview_agent, progress_agent],
)

In [12]:

# %% CELL 8 — Runner setup
APP_NAME = "esl_interview_practice"
USER_ID = "demo_learner"
SESSION_ID = "session_001"
 
runner = InMemoryRunner(agent=root_agent, app_name=APP_NAME)
session_service = runner.session_service
 
 
async def setup_session():
    await session_service.create_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID
    )
 
 
async def send(query: str) -> str:
    content = types.Content(role="user", parts=[types.Part(text=query)])
    final_text = ""
    async for event in runner.run_async(
        user_id=USER_ID, session_id=SESSION_ID, new_message=content
    ):
        if event.is_final_response() and event.content and event.content.parts:
            final_text = event.content.parts[0].text
    return final_text
 

In [13]:
async def send_with_retry(query, max_retries=4):
    for attempt in range(max_retries):
        try:
            return await send(query)
        except Exception as e:
            if "RESOURCE_EXHAUSTED" in str(e) or "429" in str(e):
                wait = 30 * (attempt + 1)
                print(f"[Rate limited — waiting {wait}s before retrying this turn...]")
                await asyncio.sleep(wait)
            else:
                raise
    raise Exception("Gave up after repeated rate limits — try again in a few minutes.")

In [14]:

# %% CELL 9 — Run a demo conversation (this generates your transcript for the notebook)
async def demo():
    await setup_session()
 
    turns = [
        "Hi, I'm ready to practice for my software engineer interview.",
        # Intentional errors for the demo transcript (article + tense):
        "I am working in tech company since three year, I am very interested for this position.",
        "Yes, I overcome a big challenge last year when my team have a deadline problem.",
        "I want work here because is good opportunity for me and my family.",
        "Wait, is this a real job offer? Should I actually take it?",  # tests guardrail
        "Sorry, let's continue practicing.",
        "I think I am good fit because I learn very fast and I am hard worker.",
        "Okay I'm done, can you give me feedback now?",
    ]
 
    for t in turns:
        print(f"\n>>> LEARNER: {t}")
        reply = await send_with_retry(t)
        print(f"<<< AGENT: {reply}")
        await asyncio.sleep(20)

In [15]:
try:
    await session_service.delete_session(app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID)
except Exception:
    pass
await demo()


>>> LEARNER: Hi, I'm ready to practice for my software engineer interview.


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.BASE_AUTHENTICATED_TOOL is enabled.
  check_feature_enabled()


<<< AGENT: 

>>> LEARNER: I am working in tech company since three year, I am very interested for this position.
<<< AGENT: That's great to hear! It sounds like you have some solid experience already.

Now, could you tell me what specifically interests you about this particular role?

>>> LEARNER: Yes, I overcome a big challenge last year when my team have a deadline problem.
<<< AGENT: Tell me more about that. What was the situation, and what did you do to address it?

>>> LEARNER: I want work here because is good opportunity for me and my family.
<<< AGENT: That's wonderful that you're thinking about your family as well.

Could you elaborate on the specific skills or experiences you have that you believe would make you a strong candidate for this software engineer position?

>>> LEARNER: Wait, is this a real job offer? Should I actually take it?
<<< AGENT: Let's pause the roleplay for a second — that sounded like a real question, not practice. I can't give real hiring, legal, or sala

![>>> LEARNER: Hi, I'm ready to practice for my software engineer interview.
<<< AGENT: Wonderful! It's great to have you here. Let's start with a common opening question to get us warmed up.

"Tell me a little bit about yourself and what brings you here today."

>>> LEARNER: I am working in tech company since three year, I am very interested for this position.
<<< AGENT: [error logged silently: article, verb tense, preposition — no visible reply, interview continues]

>>> LEARNER: Yes, I overcome a big challenge last year when my team have a deadline problem.
<<< AGENT: [error logged silently: verb tense, subject-verb agreement]

>>> LEARNER: I want work here because is good opportunity for me and my family.
<<< AGENT: [errors logged silently: infinitive, pronoun, article]

>>> LEARNER: Wait, is this a real job offer? Should I actually take it?
<<< AGENT: Let's pause the roleplay for a second — that sounded like a real question, not practice. I can't give real hiring, legal, or salary advice here. If you'd like, we can resume the practice interview, or you can ask a career counselor or trusted advisor about the real situation. Want to continue practicing?

>>> LEARNER: Sorry, let's continue practicing.
<<< AGENT: No problem at all, let's get right back to it! So, you mentioned overcoming a big challenge last year when your team had a deadline problem. Could you elaborate on that experience for me? What was the situation, what specifically was your role in addressing it, and what was the final outcome?

>>> LEARNER: I think I am good fit because I learn very fast and I am hard worker.
<<< AGENT: That's excellent to hear that you're a fast learner and a hard worker. Those are certainly valuable qualities in any role. Now, going back to the challenge you mentioned earlier – could you walk me through that situation in more detail?

>>> LEARNER: Okay I'm done, can you give me feedback now?
<<< AGENT: That was a fantastic practice session! You showed great dedication working through those questions. Let's look at a couple of things that can help you sound even more natural in your next interview.

One area where you can really shine with a small adjustment is remembering to use articles like "a" or "an" before singular job titles or descriptions. It's a common area for ESL learners.

Here are a couple of examples from your answers:
- Original: "I think I am good fit..." → Correction: "I think I am a good fit..."
- Original: "...I am hard worker." → Correction: "...I am a hard worker."

You did a wonderful job clearly stating your positive qualities like being a fast learner and a hard worker. That confidence and ability to highlight your strengths are exactly what interviewers want to hear! Keep up the great work!](http://)